<a href="https://colab.research.google.com/github/LCaravaggio/FelicidadDesigualdad/blob/main/MobilNet%20v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cargar Base

In [3]:
from google.colab import userdata
import json

!mkdir ~/.kaggle
!touch ~/.kaggle/kaggle.json

api_token = {
    'username': userdata.get('KAGGLE_USER'),
    'key': userdata.get('KAGGLE_KEY')}
with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)

!chmod 600 ~/.kaggle/kaggle.json


import kagglehub
path1 = kagglehub.dataset_download("leonardocaravaggio/ge-images")
path2 = kagglehub.dataset_download("leonardocaravaggio/ge-images2")

mkdir: cannot create directory ‘/root/.kaggle’: File exists


In [4]:
import kagglehub
import os
import shutil

# Crear una carpeta de destino
dest_folder = "imagenes"
os.makedirs(dest_folder, exist_ok=True)

# Función para copiar imágenes a una sola carpeta
def mover_imagenes(origen, destino):
    for root, _, files in os.walk(origen):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                shutil.move(os.path.join(root, file), os.path.join(destino, file))

# Copiar imágenes de ambos datasets al mismo folder
mover_imagenes(path1, dest_folder)
mover_imagenes(path2, dest_folder)

print(f"Imágenes combinadas en la carpeta: {dest_folder}")

Imágenes combinadas en la carpeta: imagenes


In [6]:
import pandas as pd
ciudades=pd.read_csv("base.csv")

In [7]:
len(ciudades)

1095

In [ ]:
import glob
import pandas as pd
from tqdm import tqdm

# Ruta donde guardaste los CSVs
ruta_features = "/content/features_por_ciudad_global"

# Buscar todos los archivos CSV
archivos_csv = glob.glob(os.path.join(ruta_features, "*.csv"))

# Leer y concatenar
df_features = pd.concat([pd.read_csv(f) for f in tqdm(archivos_csv)], ignore_index=True)
print("DataFrame unido:", df_features.shape)

100%|██████████| 2190/2190 [01:31<00:00, 24.01it/s]


DataFrame unido: (2190, 1923)


In [122]:
ciudades['Desigualdad_10km']=np.nan
ciudades['Desigualdad_1km']=np.nan
ciudades['Diferencia']=np.nan

#MobilNet v2

In [121]:
img_1k='/content/imagenes/AR_ Buenos Aires-25 de Mayo - 1K.png'
img_5k='/content/imagenes/AR_ Buenos Aires-25 de Mayo - 5K.png'
img_10k='/content/imagenes/AR_ Buenos Aires-25 de Mayo - 10K.png'
img_15k='/content/imagenes/AR_ Buenos Aires-25 de Mayo - 15K.png'
compute_inequality(img_1k, img_5k, img_10k, img_15k)

{'Desigualdad_1km': np.float32(0.60530305),
 'Desigualdad_5km': np.float32(0.62145996),
 'Desigualdad_10km': np.float32(0.60880035),
 'Desigualdad_15km': np.float32(0.53058004)}

In [120]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
import torch.nn.functional as F
import os

# Cargar MobileNetV2 hasta la capa 12
model = models.mobilenet_v2(pretrained=True)
model = torch.nn.Sequential(*list(model.features[:13]))
model.eval()

# Transformaciones de imagen
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Función para extraer características
def extract_index(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0)  # [1, 3, 224, 224]

    with torch.no_grad():
        features = model(image_tensor)  # Pasar por la red
        avg_pool = F.adaptive_avg_pool2d(features, (1, 1)).squeeze()  # [C]
        max_pool = F.adaptive_max_pool2d(features, (1, 1)).squeeze()  # [C]
        pooled_features = torch.cat((avg_pool, max_pool), dim=0)  # [2C]
        inequality_index = np.std(pooled_features.cpu().numpy())  # desigualdad

    return inequality_index

# Función para computar desigualdad a múltiples escalas
def compute_inequality(image_1km_path, image_5km_path, image_10km_path, image_15km_path):
    return {
        "Desigualdad_1km": extract_index(image_1km_path),
        "Desigualdad_5km": extract_index(image_5km_path),
        "Desigualdad_10km": extract_index(image_10km_path),
        "Desigualdad_15km": extract_index(image_15km_path),
    }

In [123]:
# Procesamiento por ciudad
for i in range(1095):
    if pd.isna(ciudades.loc[i, "Diferencia"]):
        try:
            path='/content/'
            nombre_archivo = ciudades.City[i].replace("/", ".").replace(":", "_").replace("'", "!")
            ruta_completa = os.path.join(path, "imagenes", nombre_archivo)


            img_1k = ruta_completa + " - 1K.png"
            img_5k = ruta_completa + " - 5K.png"
            img_10k = ruta_completa + " - 10K.png"
            img_15k = ruta_completa + " - 15K.png"

            if not os.path.exists(img_1k):
                print(f"❌ No existe: {img_1k}")
                continue

            if not os.path.exists(img_5k):
                print(f"❌ No existe: {img_5k}")
                continue

            if not os.path.exists(img_10k):
                print(f"❌ No existe: {img_10k}")
                continue

            if not os.path.exists(img_15k):
                print(f"❌ No existe: {img_15k}")
                continue

            resultados = compute_inequality(img_1k, img_5k, img_10k, img_15k)
            ciudades.loc[i, "Desigualdad_1km"] = resultados["Desigualdad_1km"]
            ciudades.loc[i, "Desigualdad_5km"] = resultados["Desigualdad_5km"]
            ciudades.loc[i, "Desigualdad_10km"] = resultados["Desigualdad_10km"]
            ciudades.loc[i, "Desigualdad_15km"] = resultados["Desigualdad_15km"]
            #ciudades.loc[i, "Diferencia"] = resultados["Diferencia"]

        except Exception as e:
            print(f"⚠️ Error en {ciudades.City[i]}: {e}")


In [ ]:
compute_inequality('/content/Oceano - 1K.png', '/content/Oceano - 10K.png')

{'Desigualdad_1km': np.float64(-1.8076874914858219),
 'Desigualdad_10km': np.float64(-2.110969703056616),
 'Diferencia': np.float64(0.30328221157079405)}

In [ ]:
compute_inequality('/content/Amazonas - 1K.png', '/content/Amazonas - 10K.png')

{'Desigualdad_1km': np.float64(-0.3572059270382284),
 'Desigualdad_10km': np.float64(-0.41988109745637825),
 'Diferencia': np.float64(0.06267517041814985)}

In [ ]:
compute_inequality('/content/Cochabamba - 1K.png', '/content/Cochabamba - 10K.png')

{'Desigualdad_1km': np.float64(7.005285913087494),
 'Desigualdad_10km': np.float64(-0.2553603483763971),
 'Diferencia': np.float64(7.260646261463892)}

In [ ]:
compute_inequality('/content/Vitacura - 1K.png', '/content/Vitacura - 10K.png')

{'Desigualdad_1km': np.float64(8.110935456069951),
 'Desigualdad_10km': np.float64(1.440422560851251),
 'Diferencia': np.float64(6.6705128952187)}

In [ ]:
compute_inequality('/content/Favela Rocinha - 1K.png', '/content/Favela Rocinha - 10K.png')

{'Desigualdad_1km': np.float64(8.330613437132346),
 'Desigualdad_10km': np.float64(1.5564145015622861),
 'Diferencia': np.float64(6.77419893557006)}

In [ ]:
compute_inequality('/content/Retiro - 1K.png', '/content/Retiro - 10K.png')

{'Desigualdad_1km': np.float64(9.982487218953425),
 'Desigualdad_10km': np.float64(5.454864460867292),
 'Diferencia': np.float64(4.5276227580861335)}

# Bajar la base con el indicador de desigualdad

In [ ]:
from google.colab import files
name="base_mobilv3_norm.csv"
ciudades.to_csv(name)
files.download(name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [128]:
import statsmodels.api as sm
import numpy as np

# Definir variables
X = ciudades["Desigualdad_10km"].replace([np.inf, -np.inf], np.nan)
y = ciudades["P1ST"].replace([np.inf, -np.inf], np.nan)

# Filtrar filas con NaN en X o y
mask = X.notna() & y.notna()
X, y = X[mask], y[mask]

# Agregar constante para la ordenada al origen
X = sm.add_constant(X)

# Ajustar modelo
modelo = sm.OLS(y, X).fit()

# Resumen de la regresión
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                   P1ST   R-squared:                       0.059
Model:                            OLS   Adj. R-squared:                  0.058
Method:                 Least Squares   F-statistic:                     68.96
Date:                Mon, 21 Apr 2025   Prob (F-statistic):           2.94e-16
Time:                        20:45:41   Log-Likelihood:                -454.51
No. Observations:                1095   AIC:                             913.0
Df Residuals:                    1093   BIC:                             923.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const                4.1447      0.125  